# 🎵 Aakaar AI — Unified Production Backend Server
Includes **Feature 1: MusicGen Audio Generation** and **Feature 2: Real Whisper + pYIN Hindi Lyric Note Alignment & Scale Predictor**.

In [ ]:
# Cell 1: Install Dependencies
!pip install -q transformers==4.30.2 accelerate soundfile librosa openai-whisper fastapi uvicorn pyngrok nest-asyncio python-multipart sentencepiece
print('✅ Dependencies installed!')

In [ ]:
# ============================================================
# 🎵 AAKAAR AI — UNIFIED BACKEND SERVER (FastAPI + Colab)
# Feature 1: Text-to-Music (MusicGen)
# Feature 2: Music Scale & Lyric Notes Predictor (Whisper + pYIN)
# ============================================================

import os
import re
import uuid
import gc
import json
import threading
from pathlib import Path
from collections import Counter

import torch
import numpy as np
import soundfile as sf
import librosa
import whisper

from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from pyngrok import ngrok
import uvicorn
import nest_asyncio

# Apply nest_asyncio so Uvicorn can run inside Colab's event loop
nest_asyncio.apply()

# --- 1. Hardware & Paths ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUTPUT_DIR = Path("/content/generated_audio")
TEMP_DIR = Path("/content/temp_uploads")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. Load Models Once ---
print("Loading MusicGen model...")
MUSICGEN_MODEL_NAME = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(MUSICGEN_MODEL_NAME)
music_model = MusicgenForConditionalGeneration.from_pretrained(MUSICGEN_MODEL_NAME).to(DEVICE)
music_model.eval()
SAMPLE_RATE = music_model.config.audio_encoder.sampling_rate
print("MusicGen loaded! Sample rate:", SAMPLE_RATE)

print("Loading Whisper model for lyrics recognition...")
whisper_model = whisper.load_model("small", device=DEVICE)
print("Whisper model loaded!")

# --- 3. Feature 1: Prompt Cleaning & Helpers ---
WORD_CORRECTIONS = {
    "voilen": "violin",
    "voilin": "violin",
    "violien": "violin",
    "loofy": "lo-fi",
    "lofi": "lo-fi",
    "raininy": "rainy",
    "soothng": "soothing",
    "relxing": "relaxing",
    "paino": "piano",
    "drumm": "drums",
}

def process_prompt(user_prompt: str) -> str:
    if not user_prompt or not user_prompt.strip():
        raise ValueError("Please enter a music description.")

    prompt = re.sub(r"\s+", " ", user_prompt.strip())
    for wrong, correct in WORD_CORRECTIONS.items():
        prompt = re.sub(
            rf"(?<!\w){re.escape(wrong)}(?!\w)",
            correct,
            prompt,
            flags=re.IGNORECASE,
        )

    return (
        "Instrumental music, "
        + prompt
        + ", coherent musical arrangement, high quality audio"
    )

generation_lock = threading.Lock()

# --- 4. FastAPI Setup ---
app = FastAPI(title="Aakaar AI Full Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["Content-Disposition", "Content-Length"],
)

@app.get("/")
def health():
    return {"status": "online", "features": ["text-to-music", "scale-predictor"]}

# --- 5. Feature 1 API: Text to Music ---
class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=500)
    duration: int = Field(default=5, ge=5, le=15)

@app.post("/api/generate")
def generate_music(request: GenerateRequest):
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(status_code=429, detail="A generation is already running. Please wait.")

    try:
        final_prompt = process_prompt(request.prompt)
        inputs = processor(text=[final_prompt], padding=True, return_tensors="pt")
        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

        max_new_tokens = request.duration * 50

        with torch.inference_mode():
            audio_values = music_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                guidance_scale=3.0,
            )

        audio = audio_values[0].detach().float().cpu().numpy()
        audio = np.squeeze(audio)
        if audio.ndim == 2:
            audio = audio.T

        if audio.size == 0 or not np.isfinite(audio).all():
            raise RuntimeError("Generated audio is invalid.")
        if float(np.max(np.abs(audio))) < 1e-5:
            raise RuntimeError("Generated audio is nearly silent.")

        filename = f"music_{uuid.uuid4().hex[:12]}.wav"
        path = OUTPUT_DIR / filename
        sf.write(str(path), audio, SAMPLE_RATE, subtype="PCM_16")

        return {
            "message": "Music generated successfully.",
            "prompt": final_prompt,
            "duration": request.duration,
            "audio_url": f"/api/audio/{filename}",
            "download_url": f"/api/audio/{filename}?download=true",
        }

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        raise HTTPException(status_code=503, detail="GPU memory exhausted. Try generating 5 seconds.")
    except Exception as exc:
        print("Generation error:", repr(exc))
        raise HTTPException(status_code=500, detail=str(exc))
    finally:
        generation_lock.release()

@app.get("/api/audio/{filename}")
def get_audio(filename: str, download: bool = False):
    if not re.fullmatch(r"music_[a-f0-9]{12}\.wav", filename):
        raise HTTPException(status_code=404, detail="Audio not found.")

    path = OUTPUT_DIR / filename
    if not path.is_file():
        raise HTTPException(status_code=404, detail="Audio not found.")

    return FileResponse(
        str(path),
        media_type="audio/wav",
        filename=filename if download else None,
        content_disposition_type="attachment" if download else "inline",
    )

# --- 6. Feature 2 API: Music Scale & Lyric Notes Predictor ---
@app.post("/api/analyze-scale")
async def analyze_scale(
    file: UploadFile = File(...),
    start_seconds: float = Form(0.0),
    end_seconds: float = Form(30.0),
):
    temp_id = uuid.uuid4().hex[:8]
    ext = os.path.splitext(file.filename)[1] or ".wav"
    temp_path = TEMP_DIR / f"upload_{temp_id}{ext}"

    try:
        content = await file.read()
        with open(temp_path, "wb") as f:
            f.write(content)

        # 1. Load audio with librosa
        audio, sr = librosa.load(str(temp_path), sr=22050, mono=True)
        total_duration = len(audio) / sr

        start_sec = max(0.0, float(start_seconds))
        end_sec = min(total_duration, float(end_seconds))
        if end_sec <= start_sec:
            end_sec = min(total_duration, start_sec + 30.0)
        if end_sec - start_sec > 120.0:
            end_sec = start_sec + 120.0

        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        segment = audio[start_sample:end_sample]

        if len(segment) == 0:
            raise HTTPException(status_code=400, detail="Audio segment is empty.")

        # 2. Whisper Hindi / Speech Transcription
        segment_16k = librosa.resample(segment, orig_sr=sr, target_sr=16000)
        transcription = whisper_model.transcribe(
            segment_16k,
            language="hi",
            task="transcribe",
            fp16=(DEVICE == "cuda"),
            condition_on_previous_text=False
        )

        lyrics = transcription.get("text", "").strip()
        segments = transcription.get("segments", [])
        language = transcription.get("language", "unknown")

        # 3. Pitch Detection with pYIN
        HOP_LENGTH = 256
        FRAME_LENGTH = 2048
        MIN_CONFIDENCE = 0.05
        MIN_NOTE_DURATION = 0.08
        MIN_DURATION = 0.12

        f0, voiced_flag, voiced_prob = librosa.pyin(
            segment,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
            frame_length=FRAME_LENGTH,
            hop_length=HOP_LENGTH
        )
        times = librosa.times_like(f0, sr=sr, hop_length=HOP_LENGTH)

        # 4. Convert pitch to MIDI notes
        midi_values = np.full(len(f0), np.nan)
        for i, (freq, voiced, confidence) in enumerate(zip(f0, voiced_flag, voiced_prob)):
            if voiced and np.isfinite(freq) and np.isfinite(confidence) and confidence >= MIN_CONFIDENCE:
                midi_values[i] = round(librosa.hz_to_midi(freq))

        # 5. Group consecutive frames
        note_events = []
        current_note = None
        start_time = None
        last_time = None
        frame_step = HOP_LENGTH / sr

        for i, midi in enumerate(midi_values):
            t = float(times[i])
            if np.isfinite(midi):
                midi = int(midi)
                if current_note is None:
                    current_note = midi
                    start_time = t
                    last_time = t
                elif midi == current_note and t - last_time <= frame_step * 1.5:
                    last_time = t
                else:
                    end_time = last_time + frame_step
                    if end_time - start_time >= MIN_NOTE_DURATION:
                        note_events.append({
                            "start_seconds": round(start_time + start_sec, 3),
                            "end_seconds": round(end_time + start_sec, 3),
                            "note": librosa.midi_to_note(current_note),
                            "midi_note": current_note
                        })
                    current_note = midi
                    start_time = t
                    last_time = t
            else:
                if current_note is not None:
                    end_time = last_time + frame_step
                    if end_time - start_time >= MIN_NOTE_DURATION:
                        note_events.append({
                            "start_seconds": round(start_time + start_sec, 3),
                            "end_seconds": round(end_time + start_sec, 3),
                            "note": librosa.midi_to_note(current_note),
                            "midi_note": current_note
                        })
                    current_note = None
                    start_time = None
                    last_time = None

        if current_note is not None:
            end_time = last_time + frame_step
            if end_time - start_time >= MIN_NOTE_DURATION:
                note_events.append({
                    "start_seconds": round(start_time + start_sec, 3),
                    "end_seconds": round(end_time + start_sec, 3),
                    "note": librosa.midi_to_note(current_note),
                    "midi_note": current_note
                })

        filtered_events = [
            ev for ev in note_events
            if ev["end_seconds"] - ev["start_seconds"] >= MIN_DURATION
        ]

        # 6. Detected Scale & Pitch Class Counts
        note_counts = Counter(ev["note"] for ev in filtered_events)
        most_common_pitch = note_counts.most_common(1)[0][0] if note_counts else "Unknown"

        # 7. Exact Phrase-level Alignment
        aligned_phrases = []
        structured_lines = []

        if segments:
            for item in segments:
                p_start = float(item["start"]) + start_sec
                p_end = float(item["end"]) + start_sec
                p_text = item["text"].strip()

                overlapping_notes = [
                    {
                        "note": ev["note"],
                        "start_seconds": ev["start_seconds"],
                        "end_seconds": ev["end_seconds"]
                    }
                    for ev in filtered_events
                    if (ev["start_seconds"] < p_end and ev["end_seconds"] > p_start)
                ]

                # Map words to notes
                words = p_text.split()
                line_words = []
                for idx, w in enumerate(words):
                    if idx < len(overlapping_notes):
                        assigned_note = overlapping_notes[idx]["note"]
                    elif overlapping_notes:
                        assigned_note = overlapping_notes[-1]["note"]
                    else:
                        assigned_note = "--"
                    line_words.append({"word": w, "note": assigned_note})

                aligned_phrases.append({
                    "start_seconds": round(p_start, 2),
                    "end_seconds": round(p_end, 2),
                    "lyrics": p_text,
                    "notes": overlapping_notes
                })
                if line_words:
                    structured_lines.append(line_words)
        else:
            # Fallback if instrumental
            words = []
            for ev in filtered_events[:20]:
                words.append({"word": "♪", "note": ev["note"]})
            structured_lines.append(words)

        return {
            "status": "success",
            "file_name": file.filename,
            "duration": round(total_duration, 2),
            "start_seconds": round(start_sec, 2),
            "end_seconds": round(end_sec, 2),
            "language": language,
            "lyrics": lyrics if lyrics else "[No lyrics detected / Instrumental]",
            "detected_scale": f"{most_common_pitch} Scale",
            "lines": structured_lines,
            "note_counts": dict(note_counts.most_common(10)),
            "aligned_phrases": aligned_phrases
        }

    except Exception as exc:
        print("Scale analysis error:", repr(exc))
        raise HTTPException(status_code=500, detail=str(exc))
    finally:
        if temp_path.exists():
            try:
                temp_path.unlink()
            except OSError:
                pass

# --- 7. Start ngrok & Uvicorn Server ---
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # <-- Paste your token here
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

print("\n" + "=" * 60)
print("🚀 AAKAAR AI BACKEND IS FULLY LIVE & CONNECTED!")
print("👉 BACKEND URL:", public_url)
print("=" * 60 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()
